# Query — Privacy-Preserving Membership Test

Two classes:

- **QueryServer** — server-side query protocol (Leader + Helpers)
- **QueryClient** — querier-side (pure client, no Rep3)


## QueryClient

Requires: ``add2_hash`` (party=1), ``add2_et`` (party=1).

``execute(element, ch_leader, ch_helper_a, ch_helper_b)`` runs the
full 9-step query and returns ``True`` / ``False``.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

import mpmt

client = mpmt.QueryClient(
    add2_hash=mpmt.ShrAdd2(ell=14, party=1),
    add2_et=mpmt.ShrAdd2(ell=4, party=1),
    ell_add2=14, ell_query=4,
    hf_num=3, bf_size=1000,
    hash_seed_list=[mpmt.get_key_128bits() for _ in range(3)],
)

result = client.execute(
    element=b"alice",
    ch_leader=ch_l,
    ch_helper_a=ch_a,
    ch_helper_b=ch_b,
)
assert result in (True, False)


## QueryClient Protocol Steps

1. ``share_element(element)`` → ADD2 with Leader
2. ``recv_key_share`` ← Leader shares each hash seed
3. ``hash`` + ``mod`` → ADD2 shares of BF indices
4. Send index shares to Helpers (RingTransport or Channel)
5. Receive dot shares from Helpers → ``ring_add`` → 2-of-2 share
6. ``equality_test(dot_querier, 0)`` — ADD2 ET
7. Receive Leader's ET share → ``ring_add`` → 0 or 1

Querier does **not** participate in the Rep3 ring.


## QueryServer Methods

- ``genbf_additive_share()`` — crng (all 3 servers)
- ``genbf_additive_share_helper(dpf_result)`` — crng + DPF (Helpers)
- ``reshare_into_rep3(additive_share)`` — 3-of-3 → Rep3
- ``step_dot(bf_query_sv, rep3_inst)`` — dot product + ring_conv if needed


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

srv = mpmt.QueryServer(
    party_id=0, rep3_q_inst=rep3_q, tree_cache=tc,
    add2_hash=add2_hash, add2_et=add2_et,
    dpf_dealer=dpf_d, hf_num=3, bf_size=1000,
)

additive = srv.genbf_additive_share()
sv = srv.reshare_into_rep3(additive_share=additive)
dot = srv.step_dot(bf_query_sv=sv, rep3_inst=rep3_1)
